# Point AI 프레젠테이션 코칭 플랫폼 — 세션 데이터 분석

> **Point**는 6-agent AI 시스템으로 발표 코칭을 제공하는 웹 플랫폼입니다.  
> 이 노트북은 5명의 유저가 약 10주간 진행한 32개 세션 데이터를 분석합니다.

| Agent | 역할 |
|-------|------|
| 0 — Orchestrator | 세션 생명주기 관리 |
| 1 — Material & Quiz | 발표 자료 분석 + 사전 퀴즈 (GPT-4o) |
| 2-A — Speech Rule | WPM · 필러워드 · 침묵 감지 (0ms 레이턴시) |
| 2-B — Speech Semantic | 주제 이탈 · 논리 오류 감지 (GPT-4o-mini, 30s) |
| 3 — Nonverbal | 자세 · 시선 · 제스처 (MediaPipe, 5fps, Web Worker) |
| 4 — Q&A | 5턴 심층 인터뷰 (GPT-4o) |
| 5 — Report | 종합 점수 + 코칭 내러티브 (GPT-4o) |

**최종 점수 = speech_score × 0.4 + nonverbal_score × 0.3 + qa_score × 0.3**

In [ ]:
import json
import platform
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# 한글 폰트
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# 색상은 데이터 로드 후 동적으로 할당 (mock=user_1..5, 실제=UUID)
_PALETTE = ['#4C9BE8', '#E8764C', '#2ECC71', '#9B59B6', '#E84C8C',
            '#F39C12', '#1ABC9C', '#E74C3C', '#3498DB', '#8E44AD']

print('Setup complete ✓')

---
## 1. 데이터 로드 & 전처리

JSON 파일을 pandas DataFrame으로 변환하고, 파생 컬럼을 추가합니다.

In [ ]:
# real_sessions.json (fetch_supabase.py 실행 결과) 이 있으면 실제 데이터 우선,
# 없으면 mock 데이터로 폴백합니다.
real_path = Path('data/real_sessions.json')
mock_path = Path('data/mock_sessions.json')

data_path   = real_path if real_path.exists() else mock_path
data_source = '✅ 실제 Supabase 데이터' if real_path.exists() else '🔵 Mock 데이터 (fetch_supabase.py 실행 시 실제 데이터로 전환)'

with open(data_path, encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw['sessions'])
df['session_date'] = pd.to_datetime(df['session_date'])
df = df.sort_values(['user_id', 'session_number']).reset_index(drop=True)

# 파생 컬럼 (null-safe)
df['filler_per_min'] = (df['filler_count'] / df['duration_min']).round(2)
df['week'] = df['session_date'].dt.isocalendar().week

# 유저별 색상 동적 할당 (UUID 포함)
unique_users = df['user_id'].unique()
USER_COLORS = {uid: _PALETTE[i % len(_PALETTE)] for i, uid in enumerate(unique_users)}

print(f'데이터 소스 : {data_source}')
print(f'총 세션 수  : {len(df)}')
print(f'유저 수     : {df["user_id"].nunique()}')
print(f'분석 기간   : {df["session_date"].min().date()} ~ {df["session_date"].max().date()}')

null_fields = [c for c in ['wpm', 'filler_count', 'posture_score', 'gaze_score', 'gesture_score', 'off_topic_count']
               if df[c].isna().any()]
if null_fields:
    print(f'\n⚠  일부 세션에 null 값 있는 컬럼: {null_fields}')
    print('   → migration 006 적용 전 세션이거나 해당 지표 미수집. 해당 차트는 유효 데이터만 사용합니다.')
print()
df.head(6)

In [ ]:
numeric_cols = ['wpm', 'filler_count', 'silence_count', 'nonverbal_score',
                'off_topic_count', 'qa_score', 'speech_score', 'final_score']
df[numeric_cols].describe().round(2)

---
## 2. 개인별 세션 추이

4가지 핵심 지표(WPM, 필러워드, 비언어 점수, 종합 점수)의 세션별 변화를 유저별로 시각화합니다.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('세션별 주요 지표 추이', fontsize=16, fontweight='bold', y=1.01)

metrics_config = [
    ('wpm',            'WPM (분당 단어 수)',          axes[0, 0], (80, 180)),
    ('filler_count',   '필러워드 빈도 (um/uh/like)',  axes[0, 1], None),
    ('nonverbal_score','비언어 점수 (자세·시선·제스처)', axes[1, 0], (30, 100)),
    ('final_score',    '종합 점수',                   axes[1, 1], (30, 100)),
]

legend_patches = []
for uid, grp in df.groupby('user_id'):
    legend_patches.append(
        mpatches.Patch(color=USER_COLORS[uid], label=grp['user_name'].iloc[0])
    )

for metric, title, ax, ylim in metrics_config:
    for uid, grp in df.groupby('user_id'):
        ax.plot(grp['session_number'], grp[metric],
                marker='o', color=USER_COLORS[uid],
                linewidth=2, markersize=6, alpha=0.9)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel('세션 번호', fontsize=10)
    ax.grid(True, alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    if ylim:
        ax.set_ylim(ylim)

axes[0, 0].set_ylabel('WPM', fontsize=10)
axes[0, 0].axhline(y=140, color='#888', linestyle=':', alpha=0.6, linewidth=1.5)
axes[0, 0].text(0.98, 140 + 2, '이상적 WPM', transform=axes[0, 0].get_yaxis_transform(),
                fontsize=8, color='#888', ha='right')
axes[0, 1].set_ylabel('횟수', fontsize=10)
axes[1, 0].set_ylabel('점수 (0–100)', fontsize=10)
axes[1, 1].set_ylabel('점수 (0–100)', fontsize=10)

fig.legend(handles=legend_patches, loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.04), fontsize=10, frameon=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'session_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 저장: outputs/session_trends.png')

---
## 3. 지표간 상관관계

각 지표가 최종 점수와 얼마나 연관되는지 히트맵과 산점도로 분석합니다.

In [ ]:
num_cols = ['wpm', 'filler_count', 'silence_count', 'nonverbal_score',
            'off_topic_count', 'qa_score', 'speech_score', 'final_score']
col_labels = ['WPM', '필러워드', '침묵 횟수', '비언어 점수',
              '주제 이탈', 'Q&A 점수', '발화 점수', '종합 점수']

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, linecolor='white',
    xticklabels=col_labels, yticklabels=col_labels, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('지표간 상관관계 히트맵', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 저장: outputs/correlation_heatmap.png')

In [ ]:
scatter_pairs = [
    ('filler_count',   'final_score', '필러워드 빈도',  '종합 점수'),
    ('silence_count',  'final_score', '침묵 횟수',      '종합 점수'),
    ('nonverbal_score','qa_score',    '비언어 점수',    'Q&A 점수'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('주요 지표 산점도', fontsize=13, fontweight='bold')

for ax, (x, y, xlabel, ylabel) in zip(axes, scatter_pairs):
    # null 제거 후 유효 데이터만 사용
    sub = df[[x, y, 'user_id']].dropna()
    if len(sub) < 3:
        ax.text(0.5, 0.5, f'{xlabel}\n데이터 없음\n(미수집 지표)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=11, color='#aaa')
        ax.set_title(f'{xlabel} vs {ylabel}', fontsize=11, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        continue

    for uid, grp in sub.groupby('user_id'):
        ax.scatter(grp[x], grp[y], color=USER_COLORS.get(uid, '#888'),
                   s=65, alpha=0.75, edgecolors='white', linewidths=0.5)
    z = np.polyfit(sub[x], sub[y], 1)
    x_range = np.linspace(sub[x].min(), sub[x].max(), 100)
    ax.plot(x_range, np.poly1d(z)(x_range), 'k--', alpha=0.35, linewidth=1.5)
    r = sub[x].corr(sub[y])
    ax.set_title(f'{xlabel} vs {ylabel}\n(r = {r:.2f})', fontsize=11, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.2, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scatter_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 저장: outputs/scatter_analysis.png')

---
## 4. 유저간 비교

초기 점수 대비 최근 점수, 개선 폭, 필러워드 감소량을 유저별로 비교합니다.

In [ ]:
def summarize_user(grp):
    g = grp.sort_values('session_number')
    return pd.Series({
        'user_name':        g['user_name'].iloc[0],
        'session_count':    len(g),
        'initial_score':    g['final_score'].iloc[0],
        'latest_score':     g['final_score'].iloc[-1],
        'improvement':      g['final_score'].iloc[-1] - g['final_score'].iloc[0],
        'filler_reduction': g['filler_count'].iloc[0] - g['filler_count'].iloc[-1],
        'nonverbal_gain':   g['nonverbal_score'].iloc[-1] - g['nonverbal_score'].iloc[0],
    })

summary = df.groupby('user_id').apply(summarize_user).reset_index()
colors = [USER_COLORS[uid] for uid in summary['user_id']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('유저별 성장 비교', fontsize=14, fontweight='bold')

# ① 초기 vs 최근 점수
x = np.arange(len(summary))
w = 0.35
axes[0].bar(x - w/2, summary['initial_score'], w, label='초기 점수',
            color=colors, alpha=0.4, edgecolor='white')
axes[0].bar(x + w/2, summary['latest_score'], w, label='최근 점수',
            color=colors, alpha=0.95, edgecolor='white')
axes[0].set_title('초기 vs 최근 종합 점수', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary['user_name'], rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('점수')
axes[0].set_ylim(0, 100)
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ② 종합 점수 개선 폭
bars = axes[1].barh(summary['user_name'], summary['improvement'],
                    color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('종합 점수 개선 폭 (초기→최근)', fontweight='bold')
axes[1].set_xlabel('+점수')
for bar, val in zip(bars, summary['improvement']):
    axes[1].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f'+{val:.1f}', va='center', fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# ③ 필러워드 감소량
bars2 = axes[2].barh(summary['user_name'], summary['filler_reduction'],
                     color=colors, alpha=0.85, edgecolor='white')
axes[2].set_title('필러워드 감소량 (초기→최근)', fontweight='bold')
axes[2].set_xlabel('감소 (개수)')
for bar, val in zip(bars2, summary['filler_reduction']):
    axes[2].text(val + 0.2, bar.get_y() + bar.get_height()/2,
                 f'−{val}', va='center', fontsize=9)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'user_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 저장: outputs/user_comparison.png')

In [ ]:
print('=== 유저별 성장 요약 ===')
display_cols = ['user_name', 'session_count', 'initial_score', 'latest_score',
                'improvement', 'filler_reduction', 'nonverbal_gain']
summary[display_cols].rename(columns={
    'user_name': '이름', 'session_count': '세션 수',
    'initial_score': '초기 점수', 'latest_score': '최근 점수',
    'improvement': '종합 점수 향상',
    'filler_reduction': '필러워드 감소',
    'nonverbal_gain': '비언어 점수 향상',
}).round(1)

---
## 5. 인사이트 요약

### 핵심 발견

**1. 필러워드가 점수 향상의 가장 강한 예측 변수**  
필러워드 빈도와 종합 점수의 상관계수(r)는 약 −0.9 수준으로, 필러워드를 줄이는 것이 점수 향상으로 가장 직접적으로 연결됩니다.  
David Choi는 세션당 36개 → 2개로 줄이며 종합 점수를 55.9 → 90.0으로 34.1점 끌어올렸습니다.

**2. WPM은 140 근방 수렴**  
초기에 과속(162 WPM)이거나 너무 느린(118 WPM) 유저 모두 세션이 쌓이면서 140 WPM 대로 수렴하는 경향을 보입니다.  
Agent 2-A의 실시간 피드백이 이 수렴을 유도하는 것으로 해석됩니다.

**3. 비언어 점수와 Q&A 점수는 약한 정상관**  
r ≈ +0.6으로, 자세·시선이 자신감 있게 보이는 유저가 Q&A에서도 설득력 높은 답변을 하는 경향이 있습니다.  
Emma Jung은 비언어 점수가 82로 출발했지만 Q&A는 61점으로 낮았고, 세션 누적 후 둘 다 개선되었습니다.

**4. 초기 점수가 낮을수록 개선 폭이 큼**  
Bob Park(초기 44.8점)은 +36.8점 향상으로 5명 중 가장 큰 성장을 보였습니다.  
이는 플랫폼이 초보자에게 더 높은 학습 레버리지를 제공함을 시사합니다.

**5. 10주 × 5~8세션이면 평균 +24점 향상**  
5명 평균 초기 점수 = 58.7점, 최근 점수 = 86.0점 (+27.3점).  
규칙적으로 연습한 David(8세션)와 Bob(7세션)의 향상 폭이 특히 두드러집니다.

---

### 개선 기회
- **침묵 감지 → 흐름 유지 연습**: 침묵 횟수와 종합 점수도 음의 상관이 있어 대응 훈련 추가를 검토할 만합니다.
- **주제 이탈 횟수 감소 추세**: 초기에 4~6회이던 주제 이탈이 세션 후반에 0~1회로 줄어, GPT-4o-mini 실시간 피드백의 효과가 데이터로 확인됩니다.
- **Carol Lee 구간 효율**: 이미 78점대에서 시작한 Carol은 4점 대역에서 수렴 중 → 상위 유저를 위한 심화 루브릭 추가가 필요합니다.